In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import tsfel
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display, Image

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.preprocessing.preprocessing  import llenar_val_vacios_str,llenar_val_vacios_ciclo,TsfelVars, ExtraVars,ToDummy, TeEncoder, CardinalityReducer
from src.modeling.feature_selection import feature_selection_by_constant, feature_selection_by_boruta, feature_selection_by_correlation
from src.modeling.simple_models import ChangeTrendPercentajeIdentifierWide,ConstantConsumptionClassifierWide
from src.modeling.supervised_models import LGBMModel, NNModel, LSTMNNModel
from src.helper.helper_functions import plot_roc

In [ ]:
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.2f}'.format #evita que muestre notacion cientifica
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True) #evita mostrar notacion cientifica

In [ ]:
seed = 2021
np.random.seed(seed)

# Paso 1 - Leer datos
***

Descripción de las columnas:

| Variable  | Descripción | Tipo de dato | Cardinalidad |
| :--- | :--- | :--- | :--- |
| Consumo de energía mensual | Indica el comportamiento de consumo a nivel mensual de los usuarios.  Se consideran los últimos 12 consumos.| Numérica | - |
| Actividad | Indica a qué actividad económica se dedica el usuario| Categoría | 284 |
| Tipo de Tarifa | Tarifa que tipo de tarifa se le cobra al usuario| Categoría | 47 |
| Tensión | Tensión instalada al usuario.| Categoría | 18 |
| Material instalacion | Indica tipo de material del medidor instalado| Categoría | 39 |
| Zona | Indica la ubicación geográfica a la que pertenece el usuario | Categoría | 38 |
| Target | Indica si hubo un comportamiento fraudulento o no | Numérica | 0 - 1 |
| Fecha inspección | Indica la fecha en que se inspeccionó al usuario| Fecha | - |

In [ ]:
df = pd.read_parquet('../data/df_anonimizado_02-2023.parquet')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
print("Proporcion de clase : ", 100*df.target.mean())

# Paso 2 - Particionar datos
***

In [ ]:
#Particionar por fecha
df_train = df[df.fecha_inspeccion<'2017-08-01'].copy()
df_val = df[(df.fecha_inspeccion>='2017-09-01')&(df.fecha_inspeccion<'2018-01-01')].copy()
df_test = df[df.fecha_inspeccion>='2018-01-01'].copy()

In [ ]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

In [ ]:
print("Proporcion de clase train : ", 100*df_train.target.mean())
print("Proporcion de clase validacion : ", 100*df_val.target.mean())
print("Proporcion de clase test : ", 100*df_test.target.mean())

# Paso 3 - Procesamiento de datos y construcción de modelos
***

In [ ]:
df_train.isnull().sum()

**<ins>Observación :</ins>** 

>En el conjunto de datos existen valores faltantes en las variables de consumo que son de tipos numericas y en las variables categóricas como "zona", "actividad", "tipo_tarifa" y "nivel_tension".

El tratamiento de valores faltantes fue abordado de la siguiente forma : 

- <ins>variables de consumo :</ins> se usaron los metodos ffill y bfill para propagar la observación válida hacia adelante o hacia atras.
- <ins>variables categoricas :</ins>  estas se rellenaron con una nueva categoria denominada "sin_dato".



In [ ]:
# Relleno de valores faltantes en serie de consumo.
df_train = llenar_val_vacios_ciclo(df_train, 12)

# Relleno de valores faltantes en variables categoricas
cols_fillna_sindatos = ['zona','actividad','tipo_tarifa','nivel_tension']
df_train = llenar_val_vacios_str(df_train,cols_fillna_sindatos,'sin_dato')

In [ ]:
df_train.head()

## Modelos Simples

### Regla : Cambio o disminución en el consumo de energía

>La hipotesis detras de esta regla es que si se existen decrementos bruscos de consumos entonces es un posible comportamiento anomalo.

Configuración : 
- last_base_value : indica la cantidad de periodos anteriores para comparar.
- last_eval_value : indica la cantidad de consumos a ser evaluados.
- threshold : indica la proporción de consumo.

In [ ]:
variables_consumo = [x for x in df.columns if '_anterior' in x]
last_base_value,last_eval_value,threshold = 3,1,60
trend_perc_model = ChangeTrendPercentajeIdentifierWide(last_base_value,last_eval_value,threshold)
pred = trend_perc_model.predict(df_test[variables_consumo])

In [ ]:
# Existen un 10% de usuarios en test que cumplieron con la regla.
100*pred.is_fraud_trend_perc.value_counts(normalize=True)

In [ ]:
# usuario ejemplo que cumplio con la regla
usr = 287671
df_test.loc[usr]

In [ ]:
plt.figure(figsize=(15,4))
y = df_test[variables_consumo].loc[usr].values
x = range(len(y))
plt.plot(x,y)
plt.scatter(x,y, color='red')
plt.ylim(0.0)
# plt.legend()
plt.grid(True)
plt.title("usr:" + str(usr)+" Cambio Trend último mes ")
plt.show()

### Regla : Consumos constante

> La hipotesis de esta regla es que si existen consumos constantes por periodos largos, entonces es un posible comportamiento anomalo.
- min_count_constante : indica la cantidad minima de periodos donde los consumo son constantes.


In [ ]:
min_count_constante =7
const_model = ConstantConsumptionClassifierWide(min_count_constante)
y_test_pred = const_model.predict(df_test[variables_consumo])

In [ ]:
# Existen aprox un 3% de usuarios en test que cumplieron con la regla.
100*y_test_pred.value_counts(normalize=True)

In [ ]:
# usuario ejemplo que cumplió con la regla
usr = 312935
df_test.loc[usr]

In [ ]:
plt.figure(figsize=(15,4))
y = df_test[variables_consumo].loc[usr].values
x = range(len(y))
plt.plot(x,y)
plt.scatter(x,y, color='red')
plt.ylim(0.0)
# plt.legend()
plt.grid(True)
plt.title("usr:" + str(usr)+" Consumo constante ")
plt.show()

## Modelos Supervisados

### Ingenieria de variables 

> El objetivo principal es derivar variables de las serie de consumo mensual. 

**Ejemplo :** 

    1. min, maximo, pendientes.
    2. variables estadisticas,temporaales y expectrales.
    
**Paquete :**

- [TSFEL](https://tsfel.readthedocs.io/en/latest/)
- [Ejemplo de uso](https://github.com/fraunhoferportugal/tsfel/blob/master/notebooks/TSFEL_SMARTWATCH_HAR_Example.ipynb)
- Otro paquete --> [TSFRESH](https://tsfresh.readthedocs.io/en/latest/)

En el siguiente ejemplo vemos una serie de consumo, luego con el paquete TSFEL, vamos a extrar variables estadisticas que luego lo podemos usar como variables predictoras en un modelo de supervisado.

In [ ]:
serie_consumo_anteriores = [153.0,  125.0,  117.0,  120.0,  128.0,  80.0,  105.0,  123.0,  101.0,  111.0,  99.0,  96.0]
plt.figure(figsize=(10,5))
plt.plot(serie_consumo_anteriores)
plt.xticks(range(12));

In [ ]:
cfg = tsfel.get_features_by_domain("statistical")
df_result = tsfel.time_series_features_extractor(cfg, serie_consumo_anteriores,n_jobs=-1)

In [ ]:
# Como resultados tenemos una diversidad de variables estadisticas como : 0_Max	0_Mean 0_Standard deviation	0_Variance, etc.
df_result.shape

In [ ]:
df_result[['0_Skewness','0_Kurtosis', '0_Standard deviation','0_Interquartile range', '0_Max', '0_Mean','0_Mean absolute deviation']]

### Selección de variables 

**<ins>Nota:</ins>** En este ejemplo el proceso pueda tardar mas de 5 minutos! --> puede levantar las variables seleccionadas ya calculadas

> El objetivo es seleccionar las mejores variables para entrenar los modelos.

**Metodos y Paquete :**

- [Boruta](https://pypi.org/project/Boruta/)
- [Ejemplo de uso boruta](https://towardsdatascience.com/feature-selection-with-boruta-in-python-676e3877e596)
- [Mutual Information](https://towardsdatascience.com/select-features-for-machine-learning-model-with-mutual-information-534fe387d5c8)

Este paso lo realizamos luego de extraer las nuevas variables derivadas de las series de consumo.

In [ ]:
%%time
# Este paso lo vamos hacer con una muestra del conjunto de data
variables_consumo = [x for x in df.columns if '_anterior' in x]
df_consumos = df_train[['index']+variables_consumo].head(12000)

# Construimos el pipeline de ingenieria de variables.
# TsfelVars --> Encapsula todas las funcionalidades del paquete TSFEL.
# ExtraVars --> Modulo que agrega variables extras, como cantidad de ceros seguidos en la serie de consumo y en distintas ventanas de tiempo.

pipe_feature_engeniering_consumo = Pipeline(
    [
        ("tsfel vars", TsfelVars(features_names_path=None,num_periodos= 12)),
        ("add vars3",  ExtraVars(num_periodos=3)),
        ("add vars6",  ExtraVars( num_periodos=6)),
        ("add vars12", ExtraVars(num_periodos=12)),

    ]
        )

df_features = pipe_feature_engeniering_consumo.fit_transform(df_consumos)

>Luego de crear nuevas variables vamos a aplicar los pasos para las seleccion de las variables mas importantes.

- Eliminamos varibles constantes
- Eliminamos las que esta altamente correlacionadas
- Seleccionamos con el metod boruta

In [ ]:
cols_for_feature_sel = [x for x in df_features.columns if x not in ['index'] + variables_consumo]
y_train = df_train.loc[df_features['index']].target

In [ ]:
%%time
select_by_constant = feature_selection_by_constant(df_features, y_train, cols_for_feature_sel, th=0.99)
print(f" # variables No constantes {len(select_by_constant)}")

select_by_corr = feature_selection_by_correlation(df_features, y_train, select_by_constant,method='pearson', th=0.95)
print(f" # variables No correlacionadas {len(select_by_corr)}")

select_by_boruta = feature_selection_by_boruta(df_features[select_by_constant], y_train, N=5)
print(f" # variables seleccionadas por Boruta : {len(select_by_boruta)}")


**<ins>Levantar variables ya seleccionadas :</ins>** 

In [ ]:
# select_by_boruta = pd.read_csv('../data/preprocesados/features.csv')['features'].tolist()

In [ ]:
len(select_by_boruta)

### Tratamiento de las variables categoricas

> Las variables categóricas son un desafío para los algoritmos de Machine Learning. Dado que la mayoría de ellos aceptan solo valores numéricos como entradas, necesitamos transformar las categorías en números para usarlos en el modelo.

In [ ]:
variables_categoricas = ['zona','actividad','material_instalacion','tipo_tarifa','nivel_tension']

In [ ]:
df_train[variables_categoricas].head()

El tratamieno de cada variable es el siguiente : 

- __actividad__:

*Reducción de cardinalidad y dummy:* 

> Variables categóricas a las que se le redujo la cardinalidad (Esta reducción se logra, por ejemplo, agrupando valores escasos que no tienen una presencia importante en el set de datos) y luego se les aplicó One-Hot-Encoding.


- __tipo_tarifa__:

*Reducción de cardinalidad y target encoding:*

>Variables categóricas a las que se le redujo la cardinalidad y luego se las reemplazó por una medida del efecto que podrían tener en el objetivo.

- __zona y nivel_tension__:

*Variables encodeadas:*

>Variables categóricas a las que se les ha aplicado OrdinalEncoder.

- __material_instalacion__:

*Target encoding:*

>Variables categóricas a las que se le redujo la cardinalidad y luego se las reemplazó por una medida del efecto que podrían tener en el objetivo.

Nota : [Target-encoding](https://towardsdatascience.com/dealing-with-categorical-variables-by-using-target-encoder-a0f1733a4c69) 

_Finalmente el pipeline de preprocesamiento para las variables categoricas quedo configurado como se muestra a continuacion:_

```python

pipe_actividad = Pipeline([
            ('cardinality_reducer', CardinalityReducer(threshold=0.001)),
            ('a_dummy',ToDummy(['actividad']))
        ])


pipe_tarifa = Pipeline([
            ('cardinality_reducer', CardinalityReducer(threshold=0.001)),
            ('tarifa_te',TeEncoder(['tipo_tarifa'],w=20))
        ])

vars_enc = ['zona','nivel_tension']
t_features = [
    ('var_encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), vars_enc),
    ('material_isntalacion_te', TeEncoder(['material_instalacion'],w=10), ['material_instalacion']),
    ('actividad_cr_dummy', pipe_actividad, ['actividad']),
    ('tarifa_cr_te', pipe_tarifa, ['tipo_tarifa']),
    ]

preprocessor = ColumnTransformer(transformers= t_features,remainder='passthrough')

```


### Entrenamiento y evaluación de modelos supervisados


#### Procesamos los dataset de entrenamiento, validacion y test.

**<ins>Nota:</ins>** En este ejemplo este proceso pueda tardar! --> puede levantar los dataset procesados

In [ ]:
y_train = df_train.target.copy()
df_train = df_train.drop(columns=['target'])

y_val = df_val.target.copy()
df_val = df_val.drop(columns=['target'])

y_test = df_test.target.copy()
df_test = df_test.drop(columns=['target'])

In [ ]:
# Realizamos los pasos de limpieza en los conjuntos de validacion y test.
df_val = llenar_val_vacios_ciclo(df_val, 12)
df_val = llenar_val_vacios_str(df_val,cols_fillna_sindatos,'sin_dato')

df_test = llenar_val_vacios_ciclo(df_test, 12)
df_test = llenar_val_vacios_str(df_test,cols_fillna_sindatos,'sin_dato')

In [ ]:
%%time
# Calculamos las variables derivadas de las series de consumo en los 3 conjuntos de data.
df_train = pipe_feature_engeniering_consumo.fit_transform(df_train)
df_val = pipe_feature_engeniering_consumo.transform(df_val)
df_test = pipe_feature_engeniering_consumo.transform(df_test)

**<ins>Levantar datasets procesados :</ins>** 

In [ ]:
# levantar previamente calculadas
# df_train = pd.read_parquet('../data/preprocesados/df_train_p.parquet')
# df_val = pd.read_parquet('../data/preprocesados/df_val_p.parquet')
# df_test = pd.read_parquet('../data/preprocesados/df_test_p.parquet')

In [ ]:
df_train.head()

In [ ]:
# Definimos las variables finales para el entrenamiento de los modelos.
feauture_selected = select_by_boruta
cols_for_model = variables_categoricas+variables_consumo+feauture_selected

In [ ]:
# Definimos el metodo de balanceo de clases con su correspondiente umbral y el pipeline de pre-procesamiento de variables categoricas.
param_imb_method = 'under'
sam_th = 0.2
periodo = 12
preprocesor = 4 # Pipeline de variables categoricas

In [ ]:
resulado_final = {} # para guardar todas las metricas obtenidas

#### LGBM

In [ ]:
%%time
train_lgbm_model = LGBMModel(cols_for_model,
                             hyperparams=None,
                             search_hip=True,
                             sampling_th = sam_th,
                             preprocesor_num = preprocesor,
                             sampling_method=param_imb_method)
lgbm_model = train_lgbm_model.train(df_train,y_train,df_val, y_val)

In [ ]:
y_pred_test_lgbm = lgbm_model.predict_proba(df_test[cols_for_model])[:,1]
resulado_final[f'{param_imb_method}-lgbm'] = y_pred_test_lgbm

In [ ]:
print("AUC Test:  %.3f" %  roc_auc_score(y_test,y_pred_test_lgbm))

In [ ]:
plt.figure(figsize=(8,4))
sns.distplot(y_pred_test_lgbm[y_test==0], label='0')
sns.distplot(y_pred_test_lgbm[y_test==1], label='1')
plt.xlabel('score', fontsize=16)
plt.ylabel('density', fontsize=16)
plt.legend()
plt.grid()

#### NN

In [ ]:
display(Image(filename='../img/multicapa.png', width=500, height=400))

In [ ]:
%%time
features_names = variables_categoricas + feauture_selected
spents_names = variables_consumo
train_nn_model = NNModel(features_names,spents_names,sampling_th = sam_th,preprocesor_num = preprocesor,sampling_method=param_imb_method)
rnn_model,pipe_features,pipe_spent = train_nn_model.train(df_train,y_train)

In [ ]:
X_features = pipe_features.transform(df_test[features_names])
X_spents = pipe_spent.transform(df_test[spents_names])
X_test_features = np.concatenate([X_features,X_spents],axis=1)

In [ ]:
y_pred_test_rnn = rnn_model.predict(X_test_features, batch_size=train_nn_model.BATCH_SIZE)
resulado_final[f'{param_imb_method}-ffn'] = y_pred_test_rnn

In [ ]:
print("AUC Test:  %.3f" %  roc_auc_score(y_test,y_pred_test_rnn))

In [ ]:
plt.figure(figsize=(8,4))
sns.distplot(y_pred_test_rnn[y_test==0], label='0')
sns.distplot(y_pred_test_rnn[y_test==1], label='1')
plt.xlabel('score', fontsize=16)
plt.ylabel('density', fontsize=16)
plt.legend()
plt.grid()

#### LSTM-NN

In [ ]:
display(Image(filename='../img/LSTM.png', width=700, height=600))

In [ ]:
%%time
features_names = variables_categoricas + feauture_selected
spents_names = variables_consumo
lstm_nn_model = LSTMNNModel(features_names,spents_names,sampling_th = sam_th,preprocesor_num = preprocesor,sampling_method=param_imb_method)
lstm_rnn_model,pipe_features,pipe_spent = lstm_nn_model.train(df_train,y_train)

In [ ]:
X_test_features = pipe_features.transform(df_test[features_names])
X_test_spents = pipe_spent.transform(df_test[spents_names])
X_test_spents = X_test_spents.reshape((X_test_spents.shape[0],periodo,1))

In [ ]:
y_pred_test_lstm_rnn = lstm_rnn_model.predict([X_test_spents, X_test_features],batch_size=lstm_nn_model.BATCH_SIZE)[:,0]
resulado_final[f'{param_imb_method}-lstm-ffn'] = y_pred_test_lstm_rnn

In [ ]:
print("AUC Test:  %.3f" %  roc_auc_score(y_test,y_pred_test_lstm_rnn))

In [ ]:
plt.figure(figsize=(8,4))
sns.distplot(y_pred_test_lstm_rnn[y_test==0], label='0')
sns.distplot(y_pred_test_lstm_rnn[y_test==1], label='1')
plt.xlabel('score', fontsize=16)
plt.ylabel('density', fontsize=16)
plt.legend()
plt.grid()

#### Resultados modelos superisados

In [ ]:
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
mpl.rcParams['figure.figsize'] = (10, 6)

In [ ]:
l_m_auc = []

for i,x in enumerate(resulado_final.keys()):
    plot_roc(x, y_test,resulado_final[x] , color=colors[i])
    m_auc = roc_auc_score(y_test,resulado_final[x])
    l_m_auc.append((x,m_auc))
plt.legend();

pd.DataFrame(l_m_auc, columns=['metodo','auc']).sort_values('auc',ascending=False)